# Repeatability Analysis (thesis-grade)

Loads the most recent `repeatability` run and reports ISO 9283 pose repeatability (`RP_yz`) at a single goal pose.

**Run shape**: the runner now visits one target N times. Between iterations the arm returns to the configured initial (home) pose and waits for the apriltag detector to confirm the EE marker has reached the URDF-predicted home pose within `home_tol_m` for `home_hold_frames` consecutive fresh detections (or the run aborts on `home_timeout_s`). Each visit captures exactly one detection, gated on a freshly progressed TF stamp post-settle. So the resulting cluster has one sample per cycle, and `RP_yz` over those N samples *is* the between-visit arm repeatability — gear backlash + motor zero drift + control-loop residuals — without within-visit averaging diluting it.

This is the **clean repeatability number** for the thesis: frame-independent, no URDF bias, directly comparable to ISO 9283 specs from commercial arms.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from volcaniarm_calibration.analysis import (
    latest_run, list_runs, load_run,
    repeatability_iso9283, summary, threshold_color, threshold_zone,
    WEEDING_ACCEPTABLE_MM, WEEDING_MARGINAL_MM,
)

TEST_NAME = 'repeatability'
RUN_DIR = latest_run(TEST_NAME)
run = load_run(RUN_DIR)
config, tag, fk = run['config'], run['tag'], run['fk']
print(f'run_id:         {config.get("run_id")}')
print(f'status:         {config.get("status")}  failure_reason: {config.get("failure_reason")}')
print(f'goal (y, z):    {config["goals"][0]}')
print(f'iterations:     {config["num_cycles"]}')
print(f'home tolerance: {config.get("home_tol_m", 0.02) * 1000:.1f} mm '
      f'over {config.get("home_hold_frames", 5)} fresh frames')

target = tag[tag['phase'] == 'target'].copy()
print(f'\ntarget rows:    {len(target)}  (expected = iterations)')

## ISO 9283 RP_yz across iterations (world frame)

`RP_yz = mean(d_to_centroid) + 3 * std(d_to_centroid)` over the N detected EE marker origins (one sample per iteration), expressed in the world frame Y-Z plane. Frame-independent for cluster scatter. The headline thesis number.

In [ ]:
rep = repeatability_iso9283(target, dims=('det_ee_y', 'det_ee_z'))
print(f'n samples:      {rep["n"]}')
if rep['n'] >= 2:
    print(f'centroid (y,z): ({rep["centroid"][0]:.4f}, {rep["centroid"][1]:.4f}) m  (world)')
    print(f'per-dim std:    y={rep["per_dim_std"][0]*1000:.3f} mm, '
          f'z={rep["per_dim_std"][1]*1000:.3f} mm')
    s = summary(rep['d_to_centroid'])
    print(f'\ndistance-to-centroid: {s.in_mm()}')
    print(f'\nRP_yz (ISO 9283): {rep["RP_m"]*1000:.3f} mm   '
          f'(zone: {threshold_zone(rep["RP_m"]*1000)})')
    print(f'worst-case:       {rep["worst_m"]*1000:.3f} mm   '
          f'(zone: {threshold_zone(rep["worst_m"]*1000)})')
else:
    print('\nNeed >= 2 iterations for repeatability. Re-run with num_cycles >= 5.')

## Y-Z scatter of the cluster (world frame)

Per-iteration detected EE marker origins, the cluster centroid, the ISO 9283 repeatability ring, and the application threshold rings. World-frame Y-Z, so absolute position is comparable to the commanded goal pose.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

ax.scatter(target['det_ee_y'], target['det_ee_z'], marker='x', s=80, c='steelblue',
           label=f'iterations (n={len(target)})', zorder=4)
ax.scatter(target['urdf_ee_y'], target['urdf_ee_z'], marker='+', s=80,
           c='#c79a3a', label='URDF EE', zorder=3)

if rep['n'] >= 2:
    cy, cz = rep['centroid']
    ax.scatter([cy], [cz], marker='o', s=140, facecolors='none',
               edgecolors='#2e9c4a', linewidths=2.5,
               label='cluster centroid', zorder=5)
    theta = np.linspace(0, 2 * np.pi, 200)
    ax.plot(cy + rep['RP_m'] * np.cos(theta),
            cz + rep['RP_m'] * np.sin(theta),
            color=threshold_color(rep['RP_m'] * 1000),
            linewidth=2, label=f'RP_yz = {rep["RP_m"]*1000:.2f} mm')
    for r_mm, style, lbl in (
        (WEEDING_ACCEPTABLE_MM, ':', f'acceptable ({WEEDING_ACCEPTABLE_MM:.0f} mm)'),
        (WEEDING_MARGINAL_MM, '--', f'marginal ({WEEDING_MARGINAL_MM:.0f} mm)'),
    ):
        r = r_mm / 1000
        ax.plot(cy + r * np.cos(theta), cz + r * np.sin(theta),
                color='gray', linestyle=style, linewidth=1, label=lbl)

ax.set_xlabel('y [m] (world frame)')
ax.set_ylabel('z [m] (world frame)')
ax.set_title(f'repeatability cluster  ({config["run_id"]})')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
ax.legend(loc='best', fontsize=8)
fig.tight_layout()

## CDF of distance-to-centroid

Cumulative distribution of `d_to_centroid` across the N iterations. Lets you state things like *"95% of iterations land within Y mm of the cluster centroid."* Adds the application-threshold lines for context.

In [ ]:
if rep['n'] >= 2:
    d_mm = np.sort(rep['d_to_centroid'].to_numpy() * 1000)
    cdf = np.arange(1, len(d_mm) + 1) / len(d_mm)
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    ax.step(d_mm, cdf, where='post', color='steelblue', linewidth=2)
    ax.axvline(WEEDING_ACCEPTABLE_MM, color='gray', linestyle=':', alpha=0.6,
               label=f'acceptable ({WEEDING_ACCEPTABLE_MM:.0f} mm)')
    ax.axvline(WEEDING_MARGINAL_MM, color='gray', linestyle='--', alpha=0.6,
               label=f'marginal ({WEEDING_MARGINAL_MM:.0f} mm)')
    p50 = np.percentile(d_mm, 50)
    p95 = np.percentile(d_mm, 95)
    ax.axhline(0.5, color='lightgray', alpha=0.4)
    ax.axhline(0.95, color='lightgray', alpha=0.4)
    ax.axvline(p50, color='lightgray', alpha=0.4)
    ax.axvline(p95, color='lightgray', alpha=0.4)
    ax.text(p50, 0.05, f' p50={p50:.2f} mm', va='bottom', fontsize=8)
    ax.text(p95, 0.05, f' p95={p95:.2f} mm', va='bottom', fontsize=8)
    ax.set_xlabel('distance to centroid [mm]')
    ax.set_ylabel('cumulative fraction')
    ax.set_title('CDF of d_to_centroid')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='lower right', fontsize=8)
    fig.tight_layout()
else:
    print('Need >= 2 iterations for a CDF. Re-run with num_cycles >= 5.')